# 모의면접 질문생성 모델 학습 (ICT 신입) - polyglot-ko-1.3b 버전

베이스 모델을 skt/kogpt2-base-v2(2021, 1.2억 파라미터)에서 EleutherAI/polyglot-ko-1.3b(1.3B,
10배 큼)로 바꿨다. 이유: kogpt2 기반 v1은 문장이 중간에 문형이 안 맞고 어색하게 나오는 문제가
있었는데(작은 모델+적은 데이터의 한계), 데이터(interview_qa_pairs.jsonl, 5039개)는 그대로 두고
베이스 모델만 키워서 개선을 시도한다. 학습 방식(LoRA로 어댑터만 학습) 자체는 동일 - "우리가
직접 모은 데이터로 학습시켰다"는 부분은 그대로 유지된다.

**주의할 점**
- polyglot-ko는 GPT-NeoX 아키텍처라 kogpt2(GPT2 계열)랑 LoRA `target_modules`가 다르다
  (`c_attn` 대신 `query_key_value`) - 5단계에서 반영함.
- 1.3B는 125M보다 크니까 다운로드/학습 시간이 더 걸린다 (Colab 무료 T4 기준 체감 몇 분~10분대 예상).
- pad_token이 기본으로 없어서 eos_token으로 대체 설정한다 - 4단계 참고.

**전체 단계**
1. 런타임을 GPU로 설정
2. 라이브러리 설치 (버전 고정 - kogpt2 때 겪은 디코딩 문제 재발 방지용, polyglot-ko엔 필수는
   아니지만 안전하게 그대로 유지)
3. 전처리된 데이터(`interview_qa_pairs.jsonl`) 업로드
4. polyglot-ko-1.3b 모델/토크나이저 로드
5. LoRA 설정 (GPT-NeoX용 target_modules)
6. 학습 데이터셋 구성
7. 학습
8. 테스트 (질문 생성해보기)
9. 모델 저장 (zip으로 다운로드)

**0단계 (지금 바로 할 것)**: 코랩 메뉴에서 `런타임 > 런타임 유형 변경 > 하드웨어 가속기 > T4 GPU` 선택하고 저장.

## 1. 라이브러리 설치
`transformers`: 모델/토크나이저, `peft`: LoRA(가벼운 파인튜닝), `accelerate`: 학습 속도 최적화, `datasets`: 데이터 다루기 편하게.

In [ ]:
!pip install -q "transformers==4.44.2" "tokenizers==0.19.1" "peft==0.12.0" "accelerate==0.33.0" datasets

import torch
print("GPU 사용 가능:", torch.cuda.is_available())
print("GPU 이름:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (런타임을 GPU로 바꿔줘)")

## 2. 전처리된 데이터 업로드

`interview_qa_pairs_categorized.jsonl` (AI Hub 원본 질문을 제미나이로 분류/정제한 1566개 +
제미나이가 새로 생성한 220개, 합쳐서 1786개 - job/context/question/category 다 있는 파일)을
아래 셀 실행하면 뜨는 업로드 버튼으로 그대로 올리면 된다. 매번 세션 끊길 때마다 다시 올려야 하지만,
파일 하나(수백 KB)라 zip+Drive 마운트보다 이게 더 간단하다.

In [ ]:
from google.colab import files

uploaded = files.upload()  # interview_qa_pairs_categorized.jsonl 선택
JSONL_PATH = next(iter(uploaded.keys()))
print("업로드된 파일:", JSONL_PATH)

## 3. 질문 데이터 로드

각 줄이 `{"job": ..., "context": ..., "question": ..., "category": ...}` 형태의 JSONL이라 한 줄씩 읽기만 하면 된다
(파싱은 이미 로컬에서 끝냈음 - AI Hub 원본 JSON 구조를 직접 다룰 필요 없음).

In [ ]:
import json

questions = []
with open(JSONL_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        questions.append(json.loads(line))

print(f"{len(questions)}개 질문 로드")
print(questions[0])

## 4. 모델/토크나이저 로드
SKT KoGPT2(1.2억 파라미터) - 가볍고 한국어 생성에 무난한 베이스 모델.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "EleutherAI/polyglot-ko-1.3b"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# polyglot-ko는 기본 pad_token이 없어서 eos_token으로 대체 - 없으면 배치 패딩 시 에러난다.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype="auto")
print("모델 파라미터 수:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

## 5. LoRA 설정

모델 전체(1.2억 파라미터)를 다 학습시키는 대신, 일부 레이어에 작은 "어댑터"만 붙여서 그것만 학습한다.

- 학습 속도 훨씬 빠름, GPU 메모리 훨씬 적게 씀 (무료 T4로 충분)
- `r`: 어댑터 크기 (작을수록 가볍지만 표현력 낮음, 8~16이 보통 무난)

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    # GPT-NeoX 계열(polyglot-ko)의 결합된 attention 프로젝션 모듈 이름 - GPT2의 c_attn과 역할은
    # 같지만 이름이 다르다. 잘못 넣으면 "target modules not found" 에러가 난다.
    target_modules=["query_key_value"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # 전체 대비 학습되는 파라미터 비율 확인 (보통 1% 미만)

## 6. 학습 데이터셋 구성

4단계에서 만든 `questions` 리스트를 "직무: OO\n이전 답변: ...\n카테고리: OO\n다음 질문: 실제질문" 형태의 텍스트로 바꿔서 토크나이징한다.
모델은 이 패턴을 배워서, "다음 질문:" 뒤를 이어 쓰는 방식으로 새 질문을 생성하게 된다. 카테고리를 프롬프트에 포함시켜서, 나중에 원하는 카테고리를 지정해 질문을 생성할 수 있게 한다.

In [ ]:
from datasets import Dataset

def to_prompt(item):
    job = item.get("job", "IT")
    context = item.get("context", "")
    category = item.get("category", "")
    question = item["question"]
    return f"직무: {job}\n이전 답변: {context}\n카테고리: {category}\n다음 질문: {question}{tokenizer.eos_token}"

texts = [to_prompt(q) for q in questions]  # 4단계에서 만든 questions 사용

def tokenize_fn(batch):
    out = tokenizer(batch["text"], truncation=True, max_length=288, padding="max_length")
    out["labels"] = out["input_ids"].copy()
    return out

raw_ds = Dataset.from_dict({"text": texts})
tokenized_ds = raw_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
print(tokenized_ds)

## 7. 학습
`num_train_epochs`, `per_device_train_batch_size`는 데이터 양 보고 나중에 같이 조정하자 (데이터 몇 개 나오는지에 따라 다름).

In [ ]:
from transformers import TrainingArguments, Trainer

# 1.3B는 125M보다 메모리를 더 쓰니까 per_device_train_batch_size를 줄이고 accumulation을 늘려서
# 유효 배치 크기(2*8=16)는 v1(4*4=16)과 비슷하게 유지했다.
model.gradient_checkpointing_enable()
model.enable_input_require_grads()  # LoRA + gradient checkpointing 같이 쓸 때 필요한 설정

training_args = TrainingArguments(
    output_dir="/content/checkpoints",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(model=model, args=training_args, train_dataset=tokenized_ds)
trainer.train()

## 8. 테스트 - 질문 생성해보기

In [ ]:
def generate_question(job: str, context: str = "", category: str = "") -> str:
    prompt = f"직무: {job}\n이전 답변: {context}\n카테고리: {category}\n다음 질문:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs, max_new_tokens=40, do_sample=True, top_p=0.9, temperature=0.8,
        repetition_penalty=1.3, no_repeat_ngram_size=3,
        pad_token_id=tokenizer.pad_token_id,
    )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    return text.split("다음 질문:")[-1].strip()

print(generate_question("백엔드 개발자"))
print(generate_question("프론트엔드 개발자", context="React로 SPA를 개발한 경험이 있습니다."))
print(generate_question("백엔드 개발자", category="기술_직무역량"))

## 9. 모델 저장
LoRA 어댑터만 저장하면 되니까 용량 작음 (몇십 MB 수준). 2단계에서 Drive를 마운트하지 않고 파일만 업로드하는
방식으로 갔기 때문에, 여기서도 Drive 대신 로컬(`/content/`)에 저장한 뒤 zip으로 압축해서 바로 다운로드한다.
받은 zip을 압축 풀어서 `ai-server`에 두면 나중에 거기서 이 어댑터를 불러와 쓸 수 있다.

In [ ]:
SAVE_PATH = "/content/question_generator_lora"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

import shutil
from google.colab import files

ZIP_PATH = shutil.make_archive("/content/question_generator_lora", "zip", SAVE_PATH)
print("저장 완료:", ZIP_PATH)

files.download(ZIP_PATH)  # 브라우저 다운로드 창이 뜬다